# Train Resnet50 on Imagenet with PyTorch

In [ ]:
import torch 
import torchvision
import torchvision.transforms as transforms


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# set hyperparameters
num_epochs = 10
batch_size = 64
learning_rate = 0.001

# Initialize transformations for data augmentation
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=45),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.50),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load the ImageNet Object Localization Challenge dataset
train_dataset = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'
print(f'Loading the ImageNet dataset from {train_dataset}')
train_dataset = torchvision.datasets.ImageFolder(
    root=train_dataset,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# load the ResNet50 model
model = torchvision.models.resnet50(pretrained=True)

# Freeze all layers 
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 and the final classifier 
for param in model.layer.parameters():
    param.requires_grad = True 
for param in model.fc.parameters():
    param.requires_grad = True

# Parallelize training across multiple GPUs
model = torch.nn.DataParallel(model)

# Set the model to run on device 
model = model.to(device)


# Define the loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=learning_rate
)

# Train the model ...

print(f'Training the model to ImgaeNet')
for epoch in range(num_epochs):
    for inputs, labels in train_loader:
        # Move input and label tensors to the device 
        input = inputs.to(device)
        labels = labels.to(device)

        # Zero out the optimizer
        optimizer.zero_grad()

        # Forward pass 
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

    # Print the loss for every epoch
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item(): .4f}')

print(f'Finished Training, Loss: {loss.item(): .4f} ')


Loading the ImageNet dataset from /kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'